stream_metrics.py（用于计算 iou，acc 等评估值的文件）：StreamSegMetrics 运行流程

![2.png](2.png)

![3.png](3.png)

![4.png](4.png)

In [1]:
import numpy as np
from sklearn.metrics import confusion_matrix

class _StreamMetrics(object):
    def __init__(self):
        """ Overridden by subclasses """
        raise NotImplementedError()

    def update(self, gt, pred):
        """ Overridden by subclasses """
        raise NotImplementedError()

    def get_results(self):
        """ Overridden by subclasses """
        raise NotImplementedError()

    def to_str(self, metrics):
        """ Overridden by subclasses """
        raise NotImplementedError()

    def reset(self):
        """ Overridden by subclasses """
        raise NotImplementedError()      

class StreamSegMetrics(_StreamMetrics): # 继承自_StreamMetrics
    """
    Stream Metrics for Semantic Segmentation Task
    """
    def __init__(self, n_classes):
        self.n_classes = n_classes
        self.confusion_matrix = np.zeros((n_classes, n_classes))

    def update(self, label_trues, label_preds):
        for lt, lp in zip(label_trues, label_preds):
            self.confusion_matrix += self._fast_hist( lt.flatten(), lp.flatten() )
    
    @staticmethod
    def to_str(results):
        string = "\n"
        for k, v in results.items():
            if k!="Class IoU":
                string += "%s: %f\n"%(k, v)
        
        string+='Class IoU:\n'
        for k, v in results['Class IoU'].items():
           string += "\tclass %d: %f\n"%(k, v)
        return string

    def _fast_hist(self, label_true, label_pred):
        mask = (label_true >= 0) & (label_true < self.n_classes)
        hist = np.bincount(
            self.n_classes * label_true[mask].astype(int) + label_pred[mask],
            minlength=self.n_classes ** 2,
        ).reshape(self.n_classes, self.n_classes)
        return hist

    # 在main.py里调用： metrics = StreamSegMetrics(opts.num_classes)
    # metrics.get_results()
    def get_results(self):
        """Returns accuracy score evaluation result.
            - overall accuracy
            - mean accuracy
            - mean IU
            - fwavacc
        """
        hist = self.confusion_matrix

        acc = np.diag(hist).sum() / hist.sum()
        acc_cls = np.diag(hist) / hist.sum(axis=1)
        print('acc_cls:', acc_cls)
        acc_cls = np.nanmean(acc_cls)
        iu = np.diag(hist) / (hist.sum(axis=1) + hist.sum(axis=0) - np.diag(hist))
        mean_iu = np.nanmean(iu)
        freq = hist.sum(axis=1) / hist.sum()
        fwavacc = (freq[freq > 0] * iu[freq > 0]).sum()
        cls_iu = dict(zip(range(self.n_classes), iu))

        return {
                "Overall Acc": acc,
                "Mean Acc": acc_cls,
                "FreqW Acc": fwavacc,
                "Mean IoU": mean_iu,
                "Class IoU": cls_iu,
            }
        
    def reset(self):
        self.confusion_matrix = np.zeros((self.n_classes, self.n_classes))

In [2]:
# 初始化
metrics = StreamSegMetrics(n_classes=3)

# 模拟数据（3个样本）
true_labels = [
    np.array([0, 1, 2]),  # 样本1的真实标签
    np.array([1, 1, 0]),  # 样本2的真实标签
    np.array([2, 0, 1]),  # 样本3的真实标签
]
pred_labels = [
    np.array([0, 1, 0]),  # 样本1的预测标签
    np.array([1, 0, 0]),  # 样本2的预测标签
    np.array([2, 0, 1]),  # 样本3的预测标签
]

# 更新混淆矩阵
metrics.update(true_labels, pred_labels)

# 计算并打印结果
results = metrics.get_results()
print(StreamSegMetrics.to_str(results))


acc_cls: [1.   0.75 0.5 ]

Overall Acc: 0.777778
Mean Acc: 0.750000
FreqW Acc: 0.644444
Mean IoU: 0.616667
Class IoU:
	class 0: 0.600000
	class 1: 0.750000
	class 2: 0.500000

